# Day 5 – Task 2: Exploratory Data Analysis

**Objective:** Explore employee data using summary statistics, `groupby().agg()`, correlations, business questions, and `pd.merge()`.

> This project uses a small practice dataset created for this exercise. If your trainer supplied a different dataset, replace `employee_data.csv` and adjust column names as needed.

## 1. Import libraries and load the datasets

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

folder = Path.cwd()
df = pd.read_csv(folder / "employee_data.csv")
budget = pd.read_csv(folder / "department_budget.csv")

df.head()

## 2. Check the dataset

In [ ]:
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nMissing values:")
print(df.isna().sum())

## 3. Summary statistics for key numeric columns

In [ ]:
numeric_columns = ["Age", "Monthly_Income", "Job_Satisfaction"]

summary = df[numeric_columns].agg(["mean", "median", "min", "max", "std"]).T
summary = summary.round(2)
summary

## 4. Compare groups with groupby().agg()

In [ ]:
department_summary = df.groupby("Department").agg(
    Employee_Count=("Employee_ID", "count"),
    Average_Age=("Age", "mean"),
    Average_Monthly_Income=("Monthly_Income", "mean"),
    Average_Job_Satisfaction=("Job_Satisfaction", "mean"),
    Attrition_Count=("Attrition", lambda x: (x == "Yes").sum())
).round(2)

department_summary

In [ ]:
city_gender_summary = df.groupby(["City", "Gender"]).agg(
    Employee_Count=("Employee_ID", "count"),
    Average_Income=("Monthly_Income", "mean")
).round(2)

city_gender_summary

## 5. Correlations and top 3 relationships

In [ ]:
# Convert Yes/No fields to numeric so they can be included in correlation analysis.
corr_df = df[["Age", "Monthly_Income", "Job_Satisfaction"]].copy()
corr_df["OverTime_Flag"] = df["OverTime"].map({"No": 0, "Yes": 1})
corr_df["Attrition_Flag"] = df["Attrition"].map({"No": 0, "Yes": 1})

correlation_matrix = corr_df.corr(numeric_only=True)
correlation_matrix

In [ ]:
# Show the three strongest unique relationships by absolute correlation.
upper = correlation_matrix.where(
    np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool)
)
top_three = (
    upper.stack()
    .rename("Correlation")
    .to_frame()
    .assign(Absolute_Correlation=lambda x: x["Correlation"].abs())
    .sort_values("Absolute_Correlation", ascending=False)
    .head(3)
)
top_three.round(3)

**How to interpret:** Correlation ranges from -1 to +1. A positive value means two variables tend to increase together; a negative value means one tends to decrease as the other increases. Correlation does not prove cause and effect.

## 6. Business questions with numeric answers

In [ ]:
# Question 1: Which department has the highest attrition count and rate?
attrition_by_department = df.groupby("Department").agg(
    Employees=("Employee_ID", "count"),
    Attrition_Count=("Attrition", lambda x: (x == "Yes").sum())
)
attrition_by_department["Attrition_Rate_Percent"] = (
    attrition_by_department["Attrition_Count"] /
    attrition_by_department["Employees"] * 100
).round(2)

print("Q1 - Attrition by department:")
display(attrition_by_department)
print("Highest attrition count department:",
      attrition_by_department["Attrition_Count"].idxmax())
print("Highest attrition rate department:",
      attrition_by_department["Attrition_Rate_Percent"].idxmax())

In [ ]:
# Question 2: Does overtime relate to employee attrition?
overtime_attrition = df.groupby("OverTime").agg(
    Employees=("Employee_ID", "count"),
    Attrition_Count=("Attrition", lambda x: (x == "Yes").sum())
)
overtime_attrition["Attrition_Rate_Percent"] = (
    overtime_attrition["Attrition_Count"] /
    overtime_attrition["Employees"] * 100
).round(2)

print("Q2 - Attrition by overtime:")
display(overtime_attrition)

In [ ]:
# Question 3: Which age group has the highest attrition rate?
df["Age_Group"] = pd.cut(
    df["Age"],
    bins=[0, 29, 39, 49, 100],
    labels=["18-29", "30-39", "40-49", "50+"]
)

age_attrition = df.groupby("Age_Group", observed=False).agg(
    Employees=("Employee_ID", "count"),
    Attrition_Count=("Attrition", lambda x: (x == "Yes").sum())
)
age_attrition["Attrition_Rate_Percent"] = (
    age_attrition["Attrition_Count"] /
    age_attrition["Employees"].replace(0, np.nan) * 100
).fillna(0).round(2)

print("Q3 - Attrition by age group:")
display(age_attrition)

## 7. Merge department budget table using pd.merge()

In [ ]:
merged_df = pd.merge(
    department_summary.reset_index(),
    budget,
    on="Department",
    how="left"
)

merged_df["Budget_Per_Employee"] = (
    merged_df["Annual_Budget"] / merged_df["Employee_Count"]
).round(2)

merged_df

## 8. Findings

Use the numeric outputs above to write 3 short findings in your own words. For example:

- Department ___ had the highest attrition count: ___ employees.
- The attrition rate for employees working overtime was ___%, compared with ___% for employees not working overtime.
- Age group ___ had the highest attrition rate: ___%.

Fill the blanks using the results produced by your notebook. Do not copy example numbers from another dataset.

## 9. Save merged analysis (optional)

In [ ]:
merged_df.to_csv(folder / "department_analysis_with_budget.csv", index=False)
print("Saved department_analysis_with_budget.csv")